# Chapter 6 — Lenses

Companion tutorial for **[Foundations of Computer Vision](https://visionbook.mit.edu/lenses.html)** by Torralba, Isola, and Freeman (MIT Press, 2024).

This is the lens chapter of Part II (Image Formation) answering <i>how do you let in more light without sacrificing sharpness?</i> The answer starts with Snell's law. The first section derives the **lensmaker's formula**, which turns Snell's law plus a small-angle approximation into one equation,

$$\frac{1}{a} + \frac{1}{b} = \frac{1}{f},$$

relating object distance $a$, image distance $b$, and focal length $f$.

In [ ]:
from pathlib import Path

import torch
import kornia
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch
from matplotlib.collections import LineCollection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(0)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": False,
})

# Locate the chapter's assets and image output regardless of the working directory.
_CHAPTER_DIR = Path("notebooks/CV/mit-foundations/chapter-6-lenses")
BASE_DIR = _CHAPTER_DIR if _CHAPTER_DIR.is_dir() else Path(".")
ASSETS_DIR = BASE_DIR / "assets"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(exist_ok=True)


In [ ]:
# Shared drawing helpers + a consistent book-style colour palette for Chapter 6 figures.
COLORS = {"dark": "#111111", "gray": "#555555", "blue": "#1f6fe0", "red": "#c0392b",
          "green": "#3a8b3a", "cyan": "#2997ff", "panel": "#5f7695", "normal": "#45b8c8"}

def draw_angle(ax, center, angle_a, angle_b, radius, text=None, text_radius=None, text_shift=None, fs=9.8, color="0.25", lw=0.85, z=7):
    # Arc of the smaller signed sweep from angle_a to angle_b, plus an optional label.
    delta = torch.atan2(torch.sin(angle_b - angle_a), torch.cos(angle_b - angle_a))
    a0, a1 = angle_a, angle_a + delta
    t = a0 + (a1 - a0) * torch.linspace(0.0, 1.0, 80, dtype=torch.float32)
    ax.plot((center[0] + radius * torch.cos(t)).tolist(), (center[1] + radius * torch.sin(t)).tolist(), color=color, lw=lw, zorder=z)
    if text is not None:
        text_radius = radius + torch.tensor(0.20, dtype=torch.float32) if text_radius is None else text_radius
        text_shift = torch.tensor([0.0, 0.0], dtype=torch.float32) if text_shift is None else text_shift
        mid = 0.5 * (a0 + a1)
        tp = center + text_radius * torch.stack([torch.cos(mid), torch.sin(mid)]) + text_shift
        ax.text(tp[0].item(), tp[1].item(), text, fontsize=fs, color=color, ha="center", va="center")

def lens_curves(gap, bulge, half_h, samples=320):
    # The two curved sides of a biconvex lens as (left_x, right_x, y) tensors.
    y = torch.linspace(-half_h, half_h, samples)
    profile = torch.sqrt(torch.clamp(1.0 - (y / half_h) ** 2, min=0.0))
    return -gap - bulge * profile, gap + bulge * profile, y

def surface_x_at_y(y, gap, bulge, half_h, side):
    # x of a curved lens surface at height y ("left" or "right" face).
    profile = torch.sqrt(torch.clamp(1.0 - (y / half_h) ** 2, min=0.0))
    return -gap - bulge * profile if side == "left" else gap + bulge * profile

def draw_biconvex_lens(ax, left_x, right_x, y, face="#dbe7f5", lw=1.45, alpha=0.95):
    # Filled biconvex body + black surface outlines from the surface curves.
    boundary = list(zip(left_x.tolist(), y.tolist())) + list(zip(right_x.flip(0).tolist(), y.flip(0).tolist()))
    ax.add_patch(Polygon(boundary, closed=True, facecolor=face, edgecolor="none", alpha=alpha, zorder=1))
    ax.plot(left_x.tolist(), y.tolist(), color="black", lw=lw, zorder=3)
    ax.plot(right_x.tolist(), y.tolist(), color="black", lw=lw, zorder=3)

## 6.1 — Introduction

A pinhole camera works: light from a scene passes through a small opening and forms an image on a sensor on the other side. But pinholes force a trade-off. Shrink the aperture and each scene point maps to a tight spot — the image is sharp, but so little light gets through that the image is dim. Open the aperture and more light arrives, but each scene point now spreads across many sensor pixels — the image is bright but blurry. There's no single pinhole size that's both sharp and bright.

A lens breaks the trade-off. It gathers the wide cone of light a large aperture admits and *refocuses* it back to a single point on the sensor — bright like the wide pinhole, sharp like the narrow one. The rest of the chapter is the geometry of how a lens does that.

**Figure 6.1 — Brightness/sharpness trade-offs in image formation.** Three setups, same scene at the top, same sensor at the bottom. **(a) Small pinhole:** sharp image, very dim — little light reaches the sensor. **(b) Large pinhole:** bright image, very blurry — each scene point spreads to a wide disk at the sensor. **(c) Lens:** bright AND sharp — the lens collects a wide cone of light from each scene point and refocuses it to one sensor point. (Book §6.1, Figure 6.1.)

In [ ]:
# Figure 6.1 — the three sensor images: dim (small pinhole), blurry (large), sharp (lens).
scene = torch.as_tensor(plt.imread(ASSETS_DIR / "dog_scene.png")[..., :3], dtype=torch.float32)
scene = scene / scene.max() if scene.max() > 1 else scene
dim_gain, blur_k = 0.34, 69                          # tunable: small-pinhole dimming, large-pinhole blur
chw = scene.permute(2, 0, 1)[None]
img_sharp, img_dim = scene, (scene * dim_gain).clamp(0, 1)
img_blur = kornia.filters.box_blur(chw, (blur_k, blur_k))[0].permute(1, 2, 0).clamp(0, 1)
print(f"scene {tuple(scene.shape)} | blur {blur_k}px | dim x{dim_gain}")

In [ ]:
# Figure 6.1 — ray bundles from scene to sensor for each opening (book Fig 6.1).
xs, imgs, kinds = [1.8, 5.2, 8.6], [img_dim, img_blur, img_sharp], ["pin", "pin", "lens"]
gaps = [0.035, 0.12, 0.17]                           # tunable: aperture half-widths (small/large/lens)
offs = [0.05, 0.10, 0.10]                            # off-axis slant per column
ys, ya, yr, yi, yb, p = 7.65, 5.35, 4.05, 2.35, 7.03, 0.725   # scene/aperture/sensor/image/base y + half-panel
R = (yb - yr) / (ya - yr)
dark, orange, red = COLORS["dark"], "#ff5a1f", "#ff2f2f"

fig, ax = plt.subplots(figsize=(12.8, 4.8)); ax.set(xlim=(0.35, 12.15), ylim=(0.15, 8.55)); ax.axis("off")
for xc, im, gap, off, kind in zip(xs, imgs, gaps, offs, kinds):
    ax.imshow(scene, extent=[xc - p, xc + p, ys - p, ys + p], zorder=0)
    ax.imshow(im, extent=[xc - p, xc + p, yi - p, yi + p], zorder=0)
    if kind == "lens":
        s, q, lt, lb = (xc - off, yb), (xc + off, yr), (xc - gap, ya), (xc + gap, ya)
        ax.add_patch(Polygon([s, lt, q, lb], fc=orange, ec="none", alpha=.95, zorder=2))
        yy = torch.linspace(-.22, .22, 80); xx = gap * (1 - (yy / .22) ** 2).clamp(min=0).sqrt()
        ax.add_patch(Polygon([*zip((xc - xx).tolist(), (ya + yy).tolist()), *zip((xc + xx.flip(0)).tolist(), (ya + yy.flip(0)).tolist())], fc="white", ec=dark, lw=1, zorder=4))
        rays = [[s, lt, q], [s, lb, q], [s, q]]
    else:
        q = (xc + off, yr); bl = (q[0] + (xc - gap - q[0]) * R, yb); br = (q[0] + (xc + gap - q[0]) * R, yb)
        ax.add_patch(Polygon([q, br, bl], fc=orange, ec="none", alpha=.95, zorder=2))
        rays = [[bl, q], [br, q]]
    for seg in rays: ax.plot(*zip(*seg), color=red, lw=1.3, zorder=5)
    ax.plot([xc - 1.025, xc - gap], [ya, ya], [xc + gap, xc + 1.025], [ya, ya], color=dark, lw=8.5, solid_capstyle="butt", zorder=3)
    ax.add_patch(Rectangle((xc - 1.125, yr - .12), 2.25, .12, fill=False, ec=dark, lw=1, zorder=3))
    ax.vlines(torch.linspace(xc - 1.125, xc + 1.125, 17), yr - .12, yr, color=dark, lw=.8, zorder=3)

for y, t in [(ys, "Scene"), (ya, "Aperture"), (yr - .06, "Camera\nsensor"), (yi, "Image at\nsensor")]:
    ax.text(10.55, y, t, fontsize=12, color=dark, va="center")
for xc, t in zip(xs, ["small pinhole", "large pinhole", "lens"]):
    ax.text(xc, 0.48, t, fontsize=12, color=COLORS["gray"], ha="center")
plt.savefig(IMAGES_DIR / "fig06_01.png", dpi=150, bbox_inches="tight"); plt.show()

The next section derives the geometry behind panel (c): how exactly does a lens refocus a wide cone of light from each scene point back to a single sensor point? The answer is Snell's law applied twice (once at each glass surface), with a small-angle approximation that linearizes the algebra. That's the **lensmaker's formula**, which §6.2 builds next.

## 6.2 — Lensmaker's Formula

A lens refracts light at each of its two surfaces. The lensmaker's formula compresses that two-refraction process into one equation,

$$\frac{1}{a} + \frac{1}{b} = \frac{1}{f},$$

relating object distance $a$, image distance $b$, and focal length $f$. It follows from applying Snell's law twice with the small-angle approximation $\sin\theta \approx \theta$ — the *paraxial* regime where the algebra stays linear. (Book §6.2.)

For small angles in radians, $\sin\theta \approx \theta$, and Snell's law at a glass-air interface ($n_1 = 1$, $n_2 = n$) becomes the linear $\theta_1 = n\theta_2$. The book uses this paraxial form throughout the rest of the section.

**Figure 6.3(a) — Snell's law at a flat interface.** A ray crosses from a medium with refractive index $n_1$ into a denser medium with $n_2 > n_1$ and bends toward the normal. The angles $\theta_1, \theta_2$ are measured from the normal, and $n_1 \sin\theta_1 = n_2 \sin\theta_2$ relates them. The book's panel (b), a photograph of a straw refracting in a glass of water, is the same physics in the physical world; that photo is not reproduced here. (Book §6.2, Figure 6.3.)

In [ ]:
# Figure 6.3(a) — compute the refracted ray. Snell's law at a flat interface,
# n1 sin θ1 = n2 sin θ2, gives θ2 from the incident angle; two unit direction
# vectors then place the incident and refracted rays. This is the refraction the
# figure illustrates: crossing into a denser medium bends the ray toward the normal.
n1, n2 = torch.tensor(1.0), torch.tensor(1.5)          # tunable: refractive indices (n2 > n1)
theta1 = torch.deg2rad(torch.tensor(35.0))             # tunable: incident angle from the normal
theta2 = torch.asin(n1 / n2 * torch.sin(theta1))       # Snell's law -> refracted angle
incident = torch.stack([torch.sin(theta1), -torch.cos(theta1)])   # unit dirs from the downward normal
refracted = torch.stack([torch.sin(theta2), -torch.cos(theta2)])
print(f"theta1 {torch.rad2deg(theta1):.1f}deg -> theta2 {torch.rad2deg(theta2):.1f}deg  (n1={n1:.1f}, n2={n2:.1f})")

In [ ]:
# Figure 6.3(a) — draw the interface (black), surface normal (cyan) and the bold blue
# incident+refracted rays, then annotate the angles and media. Geometry comes from the
# tensors above, so the picture always agrees with Snell's law.
L = 1.55                                                # half-extent of the drawing box
src, tip = -L * incident, L * refracted                # ray endpoints on each side of the interface
blue, cyan, dark = COLORS["blue"], COLORS["cyan"], COLORS["dark"]

fig, ax = plt.subplots(figsize=(6.0, 5.4)); ax.set(xlim=(-L, L), ylim=(-L, L)); ax.set_aspect("equal"); ax.axis("off")
ax.axhline(0, color=dark, lw=3, zorder=2)              # flat interface
ax.plot([0, 0], [L, -L], color=cyan, lw=1.6, zorder=2) # surface normal
ax.plot([src[0], 0, tip[0]], [src[1], 0, tip[1]], color=blue, lw=3, zorder=3)   # incident + refracted ray
ax.plot([0.12, 0.12, 0], [0, 0.12, 0.12], color=dark, lw=1.2, zorder=3)         # right-angle mark

ax.text(-0.16, 0.52, r"$\theta_1$", fontsize=15, color=dark, ha="center")       # angle to the normal, above
ax.text(0.20, -0.66, r"$\theta_2$", fontsize=15, color=dark, ha="center")       # angle to the normal, below
ax.text(-1.38, 0.18, r"$n_1$", fontsize=15, color=dark)
ax.text(-1.38, -0.30, r"$n_2$", fontsize=15, color=dark)
ax.text(0.14, 1.40, "surface\nnormal", fontsize=11, color=dark, ha="left", va="top")
ang = torch.rad2deg(torch.atan2(incident[1], incident[0])).item()               # tilt label along the ray
ax.text(-0.86, 1.02, "ray direction", fontsize=11, color=dark, rotation=ang, rotation_mode="anchor", ha="center")
plt.savefig(IMAGES_DIR / "fig06_03a.png", dpi=150, bbox_inches="tight")
plt.show()

**Figure 6.4(a) — Thin-lens geometry.** A point on the optical axis at distance $a$ from the lens emits rays in many directions. Those rays pass through the lens at various heights up to $c$ and converge to a single image point at distance $b$ on the other side. The angle the upper extreme ray makes with the optical axis is $\theta_1$ on the object side and $\theta_4$ on the image side. The lensmaker's formula derived in this section is the statement that for a thin lens, this convergence happens at the same $b$ for every ray emitted from the same object point. (Book §6.2, Figure 6.4(a).)

In [ ]:
# Figure 6.4(a) — compute the thin-lens conjugate geometry. The lensmaker's formula
# gives the focal length f from the glass index and surface radii; the thin-lens
# equation then fixes the image distance b for an object at distance a. This is the
# convergence the figure shows: a ray leaving the object via the lens rim (height c)
# lands back on the axis at that same b.
n, R1, R2 = torch.tensor(1.5), torch.tensor(1.4), torch.tensor(1.4)   # tunable: glass index, surface radii
a, c = torch.tensor(4.0), torch.tensor(0.85)                          # tunable: object distance, ray height at lens
f = 1.0 / ((n - 1) * (1 / R1 + 1 / R2))              # lensmaker's focal length
b = a * f / (a - f)                                  # thin-lens equation -> image distance
theta1, theta4 = torch.atan(c / a), torch.atan(c / b)                 # ray angles to the axis (object / image side)
print(f"f={f:.2f}, a={a:.1f} -> b={b:.2f}  (theta1={torch.rad2deg(theta1):.1f}deg, theta4={torch.rad2deg(theta4):.1f}deg)")

In [ ]:
# Figure 6.4(a) — draw the optical axis, thin lens, the bent ray object->rim->image,
# and the red height/distance brackets. Every position comes from a, b, c above.
A, B, C = a.item(), b.item(), c.item()
dark, blue, red = COLORS["dark"], COLORS["blue"], COLORS["red"]
lh, bulge = 1.2, 0.08                                # lens half-height and rim bulge (cosmetic)

fig, ax = plt.subplots(figsize=(12, 3.6)); ax.set(xlim=(-A - 0.5, B + 0.5), ylim=(-1.75, 1.5)); ax.axis("off")
ax.axhline(0, color=dark, lw=1.2, zorder=1)                                   # optical axis
ax.text(-1.35, 0.08, "optical axis", fontsize=11, color=dark, ha="center", va="bottom")
ly = torch.linspace(-lh, lh, 120); lx = bulge * (1 - (ly / lh) ** 2).sqrt()   # thin biconvex lens outline
ax.plot(lx.tolist(), ly.tolist(), color=dark, lw=1.6, zorder=4); ax.plot((-lx).tolist(), ly.tolist(), color=dark, lw=1.6, zorder=4)
ax.text(0, -lh - 0.12, "thin lens", fontsize=11, color=dark, ha="center", va="top")
ax.plot([-A, 0, B], [0, C, 0], color=blue, lw=1.5, zorder=3)                  # object -> lens rim (height c) -> image
ax.scatter([-A, B], [0, 0], s=25, color=dark, zorder=5)                       # object & image points

ax.plot([-0.16, -0.16], [0, C], color=red, lw=1.4, zorder=3)                  # height bracket c
for yt in (0, C): ax.plot([-0.21, -0.11], [yt, yt], color=red, lw=1.4, zorder=3)
ax.text(-0.3, C / 2, r"$c$", fontsize=13, color=dark, ha="right", va="center")
by = -0.55
for x0, x1, lab in [(-A, 0, r"$a$"), (0, B, r"$b$")]:                          # distance brackets a, b
    ax.plot([x0, x1], [by, by], color=red, lw=1.4)
    for xt in (x0, x1): ax.plot([xt, xt], [by - 0.06, by + 0.06], color=red, lw=1.4)
    ax.text((x0 + x1) / 2, by - 0.1, lab, fontsize=13, color=red, ha="center", va="top")
ax.text(-A * 0.58, 0.1, r"$\theta_1$", fontsize=14, color=dark, va="bottom")  # angle at the object
ax.text(B * 0.5, 0.1, r"$\theta_4$", fontsize=14, color=dark, va="bottom")    # angle at the image
plt.savefig(IMAGES_DIR / "fig06_04a.png", dpi=150, bbox_inches="tight")
plt.show()

The figure depicts an idealization: rays appear to bend at a single point (the center of the lens) rather than refracting twice (at each glass surface). This is the *thin-lens approximation*, which assumes the lens's physical thickness is small enough to ignore. Panel (b), drawn next, distorts the geometry to expose the actual two-surface bending that the thin-lens approximation glosses over.

**Figure 6.4(b) — The labeled thin-lens geometry.** Panel (b) follows a single ray through the same thin lens as panel (a), with the geometry deliberately distorted — two surfaces pulled apart, angles enlarged — so every label stays legible. The ray refracts at the front surface, crosses the glass, refracts again at the back, and meets the axis at the image point, turning through the four angles $\theta_1, \theta_2, \theta_3, \theta_4$. At each surface, the local tilt $\theta_S$ is set by the height $c$ at which the ray crosses (brackets $C_1 \approx C_2 \approx C$ because the lens is thin, $d \approx 0$). The object distance $a$, image distance $b$, and negligible thickness $d \approx 0$ are marked below the axis. The paraxial Snell's law at each surface gives the two relations the derivation sums:

$$n\,\theta_2 = \theta_1 + \theta_S, \qquad n\,\theta_3 = \theta_4 + \theta_S.$$

(Book §6.2, Figure 6.4(b) and Table 6.1.)

In [ ]:
# Figure 6.4(b) — set up the (deliberately distorted) two-surface geometry and derive
# every labelled angle. Object and image sit on the axis; the ray crosses each spherical
# surface at height ~c, and each surface normal points at that surface's centre of
# curvature, so the tilt θ_S falls straight out of the geometry. These are the angles the
# thin-lens derivation books in Table 6.1: n θ2 = θ1 + θ_S and n θ3 = θ4 + θ_S.
xO, xF, xB, xI = 0.0, 6.0, 11.0, 15.0        # object, front & back surface vertices, image (on axis)
Rf, Rb = 9.0, 9.0                           # tunable: surface radii (small -> visibly distorted tilt)
c1, c2 = 2.3, 2.1                           # tunable: ray heights at the front / back surface
xhf = xF + Rf * (1 - torch.cos(torch.asin(torch.tensor(c1 / Rf))))   # hit x on each spherical surface
xhb = xB - Rb * (1 - torch.cos(torch.asin(torch.tensor(c2 / Rb))))
O, I = torch.tensor([xO, 0.0]), torch.tensor([xI, 0.0])
F, B = torch.tensor([xhf, c1]), torch.tensor([xhb, c2])
Cf, Cb = torch.tensor([xF + Rf, 0.0]), torch.tensor([xB - Rb, 0.0])  # centres of curvature
ang = lambda v: torch.atan2(v[1], v[0])                              # direction angle of a 2-vector
a1, a2, a3 = ang(F - O), ang(B - F), ang(I - B)                      # the three ray segments
nf_in, nb_in = ang(Cf - F), ang(Cb - B)                             # inward surface normals (toward centres)
thetaS = ang(F - Cf) - torch.pi                                      # front surface tilt from horizontal
print(f"θ1={torch.rad2deg(a1):.0f}deg θ2={torch.rad2deg(nf_in-a2):.0f}deg θ4={torch.rad2deg(-a3):.0f}deg θS={torch.rad2deg(thetaS):.0f}deg")

In [ ]:
# Figure 6.4(b) — draw the two pulled-apart surfaces, the ray bending through them, the
# surface normals + "parallel to axis" lines, and every angle arc and distance bracket.
P = torch.pi
dark, blue, cyan, green, red = COLORS["dark"], COLORS["blue"], COLORS["cyan"], COLORS["green"], COLORS["red"]
fig, ax = plt.subplots(figsize=(13, 6)); ax.set(xlim=(-0.8, xI + 1.2), ylim=(-3.4, 4.2)); ax.set_aspect("equal"); ax.axis("off")

ax.axhline(0, color=dark, lw=1.3, zorder=1)                                   # optical axis
phi = torch.linspace(-0.46, 0.46, 80)
for xc, R, s in [(xF, Rf, 1), (xB, Rb, -1)]:                                   # the two spherical surfaces
    ax.plot((xc + s * R * (1 - torch.cos(phi))).tolist(), (R * torch.sin(phi)).tolist(), color=dark, lw=2.6, zorder=4)
ax.plot([O[0], F[0], B[0], I[0]], [O[1], F[1], B[1], I[1]], color=blue, lw=2.2, zorder=3)   # ray

for hit, nin in [(F, nf_in), (B, nb_in)]:                                      # surface normals (cyan) + parallels
    d = torch.stack([torch.cos(nin), torch.sin(nin)]) * 1.9
    ax.plot([(hit - d)[0], (hit + d)[0]], [(hit - d)[1], (hit + d)[1]], color=cyan, lw=1.1, zorder=2)
ax.plot([xF - 3.0, F[0] + 0.6], [c1, c1], color=dark, lw=0.8, zorder=2)        # parallel-to-axis at front
ax.plot([B[0] - 0.6, xB + 3.2], [c2, c2], color=dark, lw=0.8, zorder=2)        # parallel-to-axis at back

# angle arcs (center, from-angle, to-angle, radius, label) — draw_angle sweeps the minor arc
arcs = [
    (O, 0, a1, 0.9, r"$\theta_1$"), (F, P, P + a1, 0.7, r"$\theta_1$"),
    (F, P, ang(F - Cf), 0.7, r"$\theta_S$"), (F, a2, nf_in, 0.7, r"$\theta_2$"),
    (B, ang(F - B), nb_in, 0.7, r"$\theta_3$"), (B, 0, ang(B - Cb), 0.7, r"$\theta_S$"),
    (B, 0, a3, 0.7, r"$\theta_4$"), (I, P, ang(B - I), 0.9, r"$\theta_4$"),
]
for center, a, b, r, lab in arcs:
    draw_angle(ax, center, torch.as_tensor(a, dtype=torch.float32), torch.as_tensor(b, dtype=torch.float32),
               r, lab, fs=15, color=green, lw=1.4)

# right-angle marks at each hit (between surface tangent and normal)
for hit, nin in [(F, nf_in), (B, nb_in)]:
    nu = torch.stack([torch.cos(nin), torch.sin(nin)]); tu = torch.stack([-nu[1], nu[0]])
    p1, p2, p3 = hit + 0.28 * nu, hit + 0.28 * nu + 0.28 * tu, hit + 0.28 * tu
    ax.plot([p1[0], p2[0], p3[0]], [p1[1], p2[1], p3[1]], color=dark, lw=1.0, zorder=5)

# red brackets: heights C1, C2 (vertical) and distances a, d, b (below the axis)
def vbr(x, h, lab, side):
    ax.plot([x, x], [0, h], color=red, lw=1.4); [ax.plot([x-0.1, x+0.1], [yt, yt], color=red, lw=1.4) for yt in (0, h)]
    ax.text(x + side * 0.2, h / 2, lab, color=red, fontsize=13, ha="center", va="center")
def hbr(x0, x1, y, lab):
    ax.plot([x0, x1], [y, y], color=red, lw=1.4); [ax.plot([xt, xt], [y-0.12, y+0.12], color=red, lw=1.4) for xt in (x0, x1)]
    ax.text((x0+x1)/2, y - 0.35, lab, color=red, fontsize=14, ha="center", va="top")
vbr(0.9, c1, r"$C_1\approx C$", -1); vbr(xI - 0.4, c2, r"$C_2\approx C$", 1)
hbr(xO, xF, -1.6, r"$a$"); hbr(xF, xB, -1.6, r"$d\approx 0$"); hbr(xB, xI, -1.6, r"$b$")

ax.text((xF + xB) / 2, 0.16, "optical axis", color=dark, fontsize=11, ha="center")
ax.text(xF - 3.0, c1 + 0.5, "parallel to\noptical axis", color=dark, fontsize=10, ha="left", va="bottom")
ax.text(xB + 1.4, c2 + 0.5, "parallel to\noptical axis", color=dark, fontsize=10, ha="left", va="bottom")
ax.text(xI - 0.2, -2.3, "distances and angles\ndistorted for clarity of labeling", color=blue, fontsize=11, ha="right", va="top")
plt.savefig(IMAGES_DIR / "fig06_04b.png", dpi=150, bbox_inches="tight")
plt.show()

**From the diagram to the lensmaker's formula.** The book's Table 6.1 sums the four small-angle Snell relations along the ray's path. Substituting the axis angles $\theta_1 \approx c/a$ and $\theta_4 \approx c/b$, and the surface tilts $\theta_{S_1} \approx c/R_1$ and $\theta_{S_2} \approx c/R_2$, gives

$$\frac{1}{a} + \frac{1}{b} = (n-1)\left(\frac{1}{R_1} + \frac{1}{R_2}\right).$$

The height $c$ cancels on both sides — every ray from the same object point lands at the same image distance $b$, regardless of where it crosses the lens. Defining

$$\frac{1}{f} = (n-1)\left(\frac{1}{R_1} + \frac{1}{R_2}\right)$$

gives the lensmaker's formula in the target form:

$$\frac{1}{a} + \frac{1}{b} = \frac{1}{f}.$$

(Book §6.2, Figure 6.4(b) and Table 6.1.)

### From flat interface to curved surface

The angles in the lensmaker derivation are measured from each lens surface's *normal*, but the lens's surfaces aren't flat — they're spherical. Two things change with curvature. The normal at the point where a ray strikes the surface is no longer parallel to the optical axis, and the tilt of that normal depends on how far above the axis the ray hits. The next figure pins down that dependence: how the surface's tilt angle $\theta_S$ relates to the radius of curvature $R$ and the hit height $c$.

**Figure 6.5 — Relation between $R$ and $\theta_S$.** A spherical surface of radius $R$ is drawn as a full circle centered on the optical axis. A ray meets the surface at height $c$ above the axis. The slanted radius from the center to that hit point makes an angle $\theta_S$ with the horizontal axis-radius — and the same angle reappears at the surface, between the vertical reference direction and the surface normal. In the small triangle formed by the slanted radius, the horizontal axis-radius, and the vertical leg of height $c$, basic trigonometry gives $\sin\theta_S = c/R$. In the paraxial regime, this simplifies to

$$\theta_S \approx \frac{c}{R},$$

which is exactly what the `surface_angle` helper above returns, and what the lensmaker derivation in Figure 6.4(b) used at each surface. (Book §6.2, Figure 6.5.)

In [ ]:
# Figure 6.5 — relate the surface tilt θ_S to the radius R. A ray meets the spherical
# surface at height c; the radius to that point makes angle θ_S = c/R (paraxial) with the
# axis, and the tangent there is tilted by the same θ_S from vertical. That one angle is
# what the lensmaker bookkeeping calls the surface tilt.
R, c = torch.tensor(1.0), torch.tensor(0.30)      # tunable: sphere radius, ray height at the surface
thetaS = 1.15 * c / R                             # surface tilt (paraxial c/R; x1.15 enlarges it for clarity)
hit = R * torch.stack([-torch.cos(thetaS), torch.sin(thetaS)])   # point on the circle, height c above the axis
print(f"θ_S = c/R = {torch.rad2deg(c / R):.1f}deg  (c={c:.2f}, R={R:.1f})")

In [ ]:
# Figure 6.5 — the circle (spherical surface), horizontal radius, tilted radius R to the
# hit point, the height c, the tangent line, and the two equal θ_S arcs (at centre and hit).
dark, P = COLORS["dark"], torch.pi
t = torch.linspace(0, 2 * P, 400)
V, O, foot = torch.tensor([-R, 0.0]), torch.tensor([0.0, 0.0]), torch.stack([hit[0], torch.tensor(0.0)])
tangent = hit + 0.72 * torch.stack([-torch.sin(thetaS), torch.cos(thetaS)])   # tangent direction at the hit
vtop = hit + torch.tensor([0.0, 0.70])                                        # vertical reference at the hit

fig, ax = plt.subplots(figsize=(6, 6)); ax.set_aspect("equal"); ax.axis("off")
ax.plot((R * torch.cos(t)).tolist(), (R * torch.sin(t)).tolist(), color=dark, lw=1.2)
for p, q in [(V, O), (hit, O), (foot, hit), (hit, vtop), (hit, tangent)]:
    ax.plot([p[0], q[0]], [p[1], q[1]], color=dark, lw=2.0)
draw_angle(ax, O, P, P - thetaS, 0.38, r"$\theta_S$", text_radius=0.56, fs=20, color=dark, lw=1.0)
draw_angle(ax, hit, 0.5 * P, 0.5 * P - thetaS, 0.34, r"$\theta_S$", text_radius=0.50, fs=20, color=dark, lw=1.0)
ax.text(*(0.55 * (hit + O) + torch.tensor([0.0, 0.06])).tolist(), r"$R$", fontsize=22, color=dark, ha="center")
ax.text((hit[0] + 0.08).item(), (0.5 * hit[1]).item(), r"$c$", fontsize=22, color=dark, ha="left", va="center")
plt.savefig(IMAGES_DIR / "fig06_05.png", dpi=150, bbox_inches="tight")
plt.show()

With $\theta_S \approx c/R$ established, the surface-tilt relation above can drop $c$ on both sides — completing the cancellation that lets one image distance $b$ serve every ray from the same object point.

### Off-axis points

The derivation so far placed the object on the optical axis. Off-axis points need only a small extension: rotating the whole construction through a small angle $\theta_R$ adds $\theta_R$ to $\theta_S$ at each surface in Table 6.1, leaving the algebra unchanged. The lensmaker's formula still holds, and the image lands at the conjugate distance $b$ on the far side, at height $P_2 = -b\,P_1 / a$ — inverted, below the axis.

In the paraxial, thin-lens limit, this promotes the focusing property from points on an axis to whole planes: every point on the object plane at distance $a$ focuses to a corresponding point on the image plane at distance $b$, both perpendicular to the optical axis.

**Figure 6.6 — Off-axis points and the equivalent lens rotation.** The off-axis object point $P_1$ sits at height $P_1$ above the optical axis, at distance $a$ from the lens; $P_0$ marks the on-axis reference. Two rays trace the image — a parallel ray refracts through the focal point at distance $f$, and a ray through the lens center proceeds undeviated. They meet at the image point on the image plane at distance $b$. The angle $\theta_R$ marks the small rotation that makes the off-axis case equivalent to the on-axis derivation. (Book §6.2, Figure 6.6.)

In [ ]:
# Figure 6.6 — an off-axis object point images through a thin lens. Two rays fix the image:
# the ray parallel to the axis (bends through the focus) and the chief ray through the lens
# centre (undeviated). Where they cross is the off-axis image, rotated below the axis by θ_R
# — the object's angular position is the "equivalent lens rotation".
n, R1, R2 = torch.tensor(1.5), torch.tensor(1.4), torch.tensor(1.4)    # tunable: glass index, surface radii
a, P1 = torch.tensor(4.0), torch.tensor(1.05)                          # tunable: object distance, object height
f = 1.0 / ((n - 1) * (1 / R1 + 1 / R2))          # lensmaker's focal length
b = a * f / (a - f)                              # image distance (conjugate)
thetaR = torch.atan(P1 / a)                      # angle the chief ray makes with the axis
P2 = -b * torch.tan(thetaR)                      # image height (inverted, below the axis)
print(f"f={f:.2f}, b={b:.2f}, θ_R={torch.rad2deg(thetaR):.1f}deg, image height={P2:.2f}")

In [ ]:
# Figure 6.6 — axis, thin lens, the parallel ray (blue) and the chief ray (black), the
# off-axis image, θ_R at the centre, and the a / f / b distance brackets.
dark, blue, red, gray, green = COLORS["dark"], COLORS["blue"], COLORS["red"], COLORS["gray"], COLORS["green"]
lx, rx, ly = lens_curves(0.0, 0.045, 1.75)                               # thin biconvex lens surfaces
lv = surface_x_at_y(torch.tensor(0.0), 0.0, 0.045, 1.75, "left").item(); rv = -lv
A, B = a.item(), b.item()
by = -0.92; ymin = min(P2.item() - 0.28, by - 0.12); ymax = P1.item() + 0.22

fig, ax = plt.subplots(figsize=(12.8, 4.0)); ax.axis("off"); ax.set(xlim=(-A - 0.25, B + 0.2), ylim=(ymin - 0.18, ymax + 0.1))
ax.plot([-A, B], [0, 0], color=gray, lw=0.8, zorder=0)                   # optical axis
draw_biconvex_lens(ax, lx, rx, ly, face="white", lw=1.5, alpha=1.0)
ax.plot([-A, -A], [ymin, ymax], color=dark, lw=1.1); ax.plot([B, B], [P2.item() - 0.28, 0.84], color=dark, lw=1.1)  # object/image planes
ax.plot([-A, 0, B], [P1.item(), P1.item(), P2.item()], color=blue, lw=1.4)   # parallel ray -> bends through the focus
ax.plot([-A, B], [P1.item(), P2.item()], color=dark, lw=0.9)                 # chief ray through the centre
draw_angle(ax, torch.zeros(2), torch.pi - thetaR, torch.pi, 0.82, r"$\theta_R$", text_radius=0.98,
           text_shift=torch.tensor([-0.05, 0.06]), fs=16, color=green, lw=1.1)
for x0, x1, y, lab in [(-A, lv, -0.14, "a"), (rv, f.item(), -0.05, "f"), (rv, B, by, "b")]:   # a / f / b brackets
    ax.plot([x0, x1], [y, y], color=red, lw=1.3)
    for xt in (x0, x1): ax.plot([xt, xt], [y - 0.06, y + 0.06], color=red, lw=1.3)
    ax.text((x0 + x1) / 2, y - 0.14, lab, fontsize=16, color=dark, ha="center", va="center")
ax.text(-A - 0.1, P1.item(), r"$P_1$", fontsize=24, color=dark, ha="right", va="center")
ax.text(-A - 0.1, 0, r"$P_0$", fontsize=24, color=dark, ha="right", va="center")
ax.text(rv + 0.12, ymin, "thin lens", fontsize=14, color=dark, ha="left", va="top")
plt.savefig(IMAGES_DIR / "fig06_06.png", dpi=150, bbox_inches="tight")
plt.show()

With Figure 6.6, §6.2 is complete: the lensmaker's formula

$$\frac{1}{a} + \frac{1}{b} = \frac{1}{f}, \qquad \frac{1}{f} = (n-1)\left(\frac{1}{R_1} + \frac{1}{R_2}\right),$$

describes how a thin lens maps every point on an object plane at distance $a$ to a corresponding point on the image plane at distance $b$. The book's Figure 6.7, a photograph of a laser pointer swept across a lens with every ray converging to the same spot on the wall, is the same focusing property demonstrated physically; that photo isn't reproduced here.

The next section takes the lensmaker's formula and puts it to work — straight-through rays at the lens center, conjugate-point ray tracing, depth of field, concave lenses, and a Galilean telescope built from two of them.

## 6.3 — Imaging with Lenses

With the lensmaker's formula in hand, the rest of the chapter puts it to work — predicting where images form, how sharp they are, what changes with a concave lens, and how two lenses combine into a telescope. The starting observation is a small one, but it's what lets the whole apparatus mimic a pinhole camera.

At the very center of a thin lens, the front and back surfaces are parallel. A ray crossing that region refracts at the first surface, then refracts back through the same angle at the second surface — emerging parallel to the way it entered, displaced laterally by a small amount that depends on the lens's thickness. In the *thin*-lens limit, the two surfaces collapse to a single plane: the displacement vanishes and the ray passes straight through. Every direction through the center is undeviated, which means the lens behaves, for those center rays, exactly like a pinhole — and a thin lens therefore images the world in *perspective projection*, just as a pinhole does.

**Figure 6.8 — Rays through the center of a thin lens.** **(a)** A physical lens of non-zero thickness: the ray refracts at both surfaces and exits parallel to the incoming direction, with a small lateral displacement. **(b)** The same ray under the thin-lens approximation: the two surfaces collapse to a single plane, the displacement vanishes, and the ray passes straight through. **(c)** Because *every* direction through the center is undeviated, a fan of rays through the lens center behaves identically to a fan of rays through a pinhole. (Book §6.3, Figure 6.8.)

In [ ]:
# Figure 6.8 — the centre ray of a lens. At the lens centre the two surfaces are parallel,
# so a ray entering at θ1 refracts in, crosses, and refracts back out at the SAME angle θ1:
# a thick lens shifts it sideways (a); a thin lens (surfaces coincident) passes it straight
# through, undeviated (b). Snell's law at the vertical faces gives the in-glass direction.
theta1 = torch.deg2rad(torch.tensor(24.0))        # tunable: incidence angle at the lens centre
n_air, n_glass = 1.0, 1.5                          # tunable: refractive indices
phi2 = torch.asin(n_air / n_glass * torch.cos(theta1))          # in-glass angle from the horizontal normal
d_air = torch.stack([torch.sin(theta1), -torch.cos(theta1)])    # incident / exit ray direction (down-right)
d_glass = torch.stack([torch.cos(phi2), -torch.sin(phi2)])      # bent ray direction inside the glass
print(f"θ1={torch.rad2deg(theta1):.0f}deg, in-glass {torch.rad2deg(0.5*torch.pi - phi2):.0f}deg from vertical")

In [ ]:
# Figure 6.8 — three panels: (a) the ray steps sideways through a thick-lens centre,
# (b) it passes straight through a thin-lens centre, (c) a fan of centre rays, all undeviated
# (so the thin-lens centre acts like a pinhole). Surfaces drawn as cyan verticals.
dark, blue, cyan, pg = COLORS["dark"], COLORS["blue"], COLORS["cyan"], COLORS["panel"]
th = 0.72                                                        # drawn thickness of the physical lens
hitL = torch.tensor([-th / 2, 0.18]); hitR = hitL + ((th) / d_glass[0]) * d_glass   # glass entry / exit points
srcA, dstA = hitL - 1.55 * d_air, hitR + 1.65 * d_air
srcB, dstB = -1.55 * d_air, 1.65 * d_air
rot = torch.rad2deg(torch.atan2(d_air[1], d_air[0])).item()
box = dict(facecolor="white", edgecolor="none", pad=0.12, alpha=0.9)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax in axs: ax.set(xlim=(-1.95, 1.95), ylim=(-1.6, 2.15)); ax.axis("off")
# (a) thick lens: two surfaces, ray air -> glass -> air (parallel shift)
axs[0].plot([-th/2]*2, [-0.95, 1.15], color=cyan, lw=1.6); axs[0].plot([th/2]*2, [-0.95, 1.15], color=cyan, lw=1.6)
axs[0].plot([srcA[0], hitL[0], hitR[0], dstA[0]], [srcA[1], hitL[1], hitR[1], dstA[1]], color=blue, lw=2.4)
axs[0].text(0.55, 1.9, "Center of\nphysical lens", ha="center", fontsize=15)
aA = hitL - 0.95*d_air + torch.tensor([0.14, 0.05]); axs[0].text(aA[0].item(), aA[1].item(), "ray direction", rotation=rot, fontsize=12, color=dark)
axs[0].text((hitL[0]-0.18).item(), (0.5*(hitL[1]+srcA[1])+0.06).item(), r"$\theta_1$", fontsize=17, bbox=box)
axs[0].text((hitR[0]+0.02).item(), (0.5*(hitR[1]+dstA[1])-0.06).item(), r"$\theta_1$", fontsize=17, bbox=box)
axs[0].text(0, -1.48, "(a)", ha="center", fontsize=19, color=pg, style="italic")
# (b) thin lens: one surface, ray straight through (undeviated)
axs[1].plot([0, 0], [-0.95, 1.15], color=cyan, lw=1.8)
axs[1].plot([srcB[0], dstB[0]], [srcB[1], dstB[1]], color=blue, lw=2.4)
axs[1].text(0.55, 1.9, "Center of\nthin lens", ha="center", fontsize=15)
aB = -0.62*d_air + torch.tensor([0.16, 0.05]); axs[1].text(aB[0].item(), aB[1].item(), "ray direction", rotation=rot, fontsize=12, color=dark)
axs[1].text(-0.42, 0.28, r"$\theta_1$", fontsize=17, bbox=box); axs[1].text(0.12, -0.42, r"$\theta_1$", fontsize=17, bbox=box)
axs[1].text(0, -1.48, "(b)", ha="center", fontsize=19, color=pg, style="italic")
# (c) every direction through the thin-lens centre is undeviated -> behaves like a pinhole
lx, rx, ly = lens_curves(0.018, 0.030, 0.92)
for a in torch.deg2rad(torch.linspace(-22, 22, 5)):
    dv = 1.75 * torch.stack([torch.cos(a), torch.sin(a)])
    axs[2].plot([-dv[0].item(), dv[0].item()], [-dv[1].item(), dv[1].item()], color=blue, lw=2.0)
draw_biconvex_lens(axs[2], lx, rx, ly, face="white", lw=1.4, alpha=1.0)
axs[2].text(0, 1.9, "thin lens", ha="center", fontsize=15)
axs[2].text(0, -1.48, "(c)", ha="center", fontsize=19, color=pg, style="italic")
plt.savefig(IMAGES_DIR / "fig06_08abc.png", dpi=150, bbox_inches="tight")
plt.show()

With this observation in place, four properties now characterize how rays travel through a thin lens:

1. **Focusing** (§6.2): every ray from a point at distance $a$ reconverges at the conjugate distance $b$, with $\frac{1}{a} + \frac{1}{b} = \frac{1}{f}$.
2. **Parallel rays**: the limit $a \to \infty$ — parallel rays converge at the focal point, distance $f$ behind the lens.
3. **Center rays**: any ray through the lens center proceeds in a straight line, as panel (c) above shows.
4. **Magnification**: combining the first and third, a plane at distance $a$ images with scale $b/a$ — exactly as a pinhole at the lens center would project it.

Two points on opposite sides of the lens at distances $a$ and $b$ that satisfy the lensmaker's formula are called **conjugate points**: light from one focuses to the other, and vice versa. The next figure traces conjugate-point pairs for several object distances, showing how the image distance moves as the object slides toward or away from the lens.

The four ray-tracing rules above pin down the *image distance* for any object distance through $\frac{1}{a} + \frac{1}{b} = \frac{1}{f}$. The two endpoints of the formula are limits — object at infinity, image at $f$; object at $f$, image at infinity — and everything in between trades smoothly between them. The next figure traces five cases on the same lens, holding the focal length fixed and moving the object closer step by step.

**Figure 6.9 — Conjugate points for a convex thin lens.** All five panels show the same lens with focal points marked at $\pm f$ from the lens (cyan dots). **(a)** Parallel rays from infinity converge at the right-side focal point — the limiting case $a \to \infty$, $b = f$. **(b)** Object at $a = 3f$ images at $b = 1.5f$. **(c)** Object at $a = 2f$ images symmetrically at $b = 2f$ — the unit-magnification case. **(d)** Object at $a = 1.5f$ images at $b = 3f$ — as the object moves inward, the image moves outward. **(e)** Object at the left focal point ($a = f$) produces parallel rays on exit — the other limiting case, $b \to \infty$. (Book §6.3, Figure 6.9.)

In [ ]:
# Figure 6.9 — conjugate points of a convex thin lens. For an object at distance a the
# thin-lens equation fixes the image at b = a·f/(a−f). Five cases sweep a from ∞ down to f,
# so the image runs from the focus out to ∞ — the standard conjugate pairs for placing images.
n, R1, R2, c = torch.tensor(1.5), torch.tensor(1.4), torch.tensor(1.4), 0.55   # tunable: glass, radii, ray height
f = (1.0 / ((n - 1) * (1 / R1 + 1 / R2))).item()      # lensmaker's focal length
img = lambda a: a * f / (a - f)                       # thin-lens equation -> image distance
print(f"f={f:.2f}; images for a=3f,2f,1.5f -> {img(3*f):.2f}, {img(2*f):.2f}, {img(1.5*f):.2f}")

In [ ]:
# Figure 6.9 — one panel per conjugate case: the thin lens, its two focal points (cyan dots),
# and three rays (heights +c, 0, −c) from the object, refracting at the lens, to the image.
# Parallel-in (a) and parallel-out (e) are the a→∞ and a→f limits.
dark, blue, cyan, pg = COLORS["dark"], COLORS["blue"], COLORS["cyan"], COLORS["panel"]
lx, rx, ly = lens_curves(0.0, 0.055, max(0.85, 1.55 * c))
h = max(0.85, 1.55 * c) + 0.3
# (tag, left-x, left-parallel?, right-x, right-parallel?, xlim)
cases = [("(a)", -2.35*f, True, f, False, (-2.35*f-.45, 1.8*f+.45)),
         ("(b)", -3*f, False, img(3*f), False, (-3*f-.45, img(3*f)+.45)),
         ("(c)", -2*f, False, 2*f, False, (-2*f-.45, 2*f+.45)),
         ("(d)", -1.5*f, False, img(1.5*f), False, (-1.5*f-.45, img(1.5*f)+.45)),
         ("(e)", -f, False, 2.35*f, True, (-1.55*f-.45, 2.35*f+.45))]

fig, axs = plt.subplots(1, 5, figsize=(18, 4.4)); fig.subplots_adjust(wspace=0.22)
for ax, (tag, Lx, Lpar, Rx, Rpar, xlim) in zip(axs, cases):
    ax.plot([-4, 4], [0, 0], color=dark, lw=0.9, zorder=0)                       # optical axis
    draw_biconvex_lens(ax, lx, rx, ly, face="white", lw=1.1, alpha=1.0)
    ax.scatter([-f, f], [0, 0], s=18, color=cyan, zorder=4)                      # focal points
    for y in (c, 0.0, -c):
        ax.plot([Lx, 0, Rx], [y if Lpar else 0, y, y if Rpar else 0], color=blue, lw=1.1, zorder=2)
    ax.text(0.5, -0.14, tag, transform=ax.transAxes, fontsize=20, color=pg, ha="center", va="top", style="italic")
    ax.set(xlim=xlim, ylim=(-h, h)); ax.axis("off")
plt.savefig(IMAGES_DIR / "fig06_09.png", dpi=150, bbox_inches="tight")
plt.show()

Set the lens up inside a camera. The sensor sits at a fixed distance $b$ behind the lens; the object lives somewhere out in the world at distance $a$. If $a$ and $b$ happen to satisfy the lensmaker's formula, the object is in *focus* — every ray from a surface point converges to a single point on the sensor, and the image is sharp.

But $a$ varies with what the camera is pointed at, and $b$ is fixed by the camera's geometry. When $a$ doesn't quite match the conjugate of $b$, rays from a single object point don't converge at the sensor — they hit it as a small *circle of confusion*. How small can that circle get before the image looks blurry, and how much of the scene stays acceptably sharp at once? That's the **depth of field**, and it's what §6.3.1 derives next.

### 6.3.1 — Depth of Field

When the lens lives inside a camera, the sensor sits at a fixed distance behind it, so only one object distance is in sharp focus — the one whose conjugate is the sensor itself. That object distance defines an *object plane* (the *focal plane* in the book's terminology) where everything images sharply. Objects in front of or behind that plane image past or before the sensor, hitting it as a small disk — a *circle of confusion*. A photograph still looks sharp as long as that circle is small enough that the eye can't tell, and the range of object distances over which it stays acceptable is the *depth of field*.

**Figure 6.10 — Circle of confusion and depth of field.** Three object positions, one fixed sensor plane. The top row shows an object on the in-focus plane — every ray converges to a single point on the sensor for a sharp image. The middle row shows an object closer than the in-focus plane — its image forms past the sensor, hitting the sensor as a circle of confusion. The bottom row shows an object further than the in-focus plane — its image forms before the sensor, again hitting as a circle of confusion. The bracket on the left marks the depth of field. (Book §6.3.1, Figure 6.10.)

In [ ]:
# Figure 6.10 — depth of field. The lens is focused so an object at focus_a images exactly on
# the sensor. Objects nearer or farther image in front of / behind the sensor, so their cones
# cross the sensor over a finite blur disc — the circle of confusion. The band of object depths
# whose blur stays acceptable is the depth of field.
n, R1, R2 = torch.tensor(1.5), torch.tensor(1.4), torch.tensor(1.4)   # tunable: glass, radii
focus_a, near_a, far_a, ap = 4.0, 3.35, 5.25, 0.82                    # tunable: focus/near/far depths, aperture radius
f = (1.0 / ((n - 1) * (1 / R1 + 1 / R2))).item()
img = lambda a: a * f / (a - f)                  # thin-lens equation -> image distance
sensor_x = img(focus_a)                          # sensor sits at the in-focus image distance
print(f"f={f:.2f}, sensor at b={sensor_x:.2f}; near/far images -> {img(near_a):.2f}/{img(far_a):.2f}")

In [ ]:
# Figure 6.10 — three object depths against one sensor plane. Each object sends a cone through
# the lens rim (±ap) to its image; where that cone meets the sensor is the circle of confusion.
dark, blue, cyan, gray = COLORS["dark"], COLORS["blue"], COLORS["cyan"], COLORS["gray"]
lx, rx, ly = lens_curves(0.0, 0.045, 0.84)
rows = [(2.55, focus_a), (0.05, near_a), (-2.45, far_a)]                         # (row y-offset, object depth)
yint = lambda hy, im, xs: hy + xs / im[0] * (im[1] - hy)                         # ray height at x=xs (ray from (0,hy)->im)

fig, ax = plt.subplots(figsize=(7.2, 8.2)); ax.set_aspect("equal"); ax.axis("off")
xmin, xmax, ymin, ymax = -far_a - 0.38, sensor_x + 0.95, -4.0, 3.9
ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
for x, lab, ha in [(-focus_a, "object plane", "right"), (sensor_x, "sensor plane", "left")]:
    ax.plot([x, x], [ymin, ymax], color=cyan, lw=0.8, alpha=0.45, zorder=0)
    ax.text(x, ymax - 0.18, lab, fontsize=10, color=dark, ha=ha, va="bottom")
cocs = []
for y0, a in rows:
    draw_biconvex_lens(ax, lx, rx, ly + y0, face="white", lw=1.2, alpha=1.0)
    im = (img(a), y0); ys = []
    for hy in (y0 + ap, y0 - ap):
        yse = yint(hy, im, sensor_x)
        ax.plot([-a, 0, im[0], sensor_x], [y0, hy, y0, yse], color=blue, lw=1.0, zorder=2)
        ys.append(yse)
    cocs.append((min(ys), max(ys)))
for ylo, yhi in cocs[1:]:                                                        # circle-of-confusion brackets
    ax.plot([sensor_x, sensor_x], [ylo, yhi], color=blue, lw=1.0, zorder=3)
    bx = sensor_x + 0.18
    for yy in (ylo, yhi): ax.plot([sensor_x, bx], [yy, yy], color=gray, lw=0.8, ls=":")
    ax.plot([bx, bx], [ylo, yhi], color=gray, lw=0.8, ls=":")
    ax.text(bx + 0.12, 0.5 * (ylo + yhi), "circle of\nconfusion", fontsize=10, color=dark, ha="left", va="center")
dy, nx, fx = 0.05 - 1.10, -near_a, -far_a                                        # depth-of-field bracket
for x0, y0 in [(fx, -2.45), (nx, 0.05)]: ax.plot([x0, x0], [dy, y0], color=gray, lw=0.8, ls=":")
ax.plot([fx, nx], [dy, dy], color=dark, lw=0.9)
for xe in (fx, nx): ax.plot([xe, xe], [dy - 0.11, dy + 0.11], color=dark, lw=0.9)
ax.text(0.5 * (fx + nx), dy + 0.02, "depth of field", fontsize=10, color=dark, ha="center", va="bottom")
plt.savefig(IMAGES_DIR / "fig06_10.png", dpi=150, bbox_inches="tight")
plt.show()

The depth of field is set by the lens's focal length, the maximum circle-of-confusion size that still looks sharp, and the camera's *f-number* $N = f / A$, where $A$ is the aperture diameter. The next figure lays out the variables.

The geometry that turns the circle of confusion into a depth-of-field range is two pairs of similar triangles — one for the near limit, one for the far.

**Figure 6.11 — Variables for the depth-of-field calculation.** Two scenes, same focal length, same focused object distance $U$, same circle of confusion $C$ at the sensor — only the aperture differs. The *f-number* $N = f/A$ relates aperture diameter $A$ to focal length: smaller $N$ means a wider aperture. **(Top, $N^a$)** Wider aperture, narrower depth of field — the range $D_1^a$ to $D_2^a$ around $U$ is short. **(Bottom, $N^b$)** Narrower aperture, wider depth of field — same $C$ at the sensor, but a much larger range $D_1^b$ to $D_2^b$ around $U$. (Book §6.3.1, Figure 6.11.)

In [ ]:
# Figure 6.11 — the variables behind the depth-of-field formula, for two apertures. A lens
# focused at distance U puts the sharp image at s; an object at depth D blurs to a disc that
# just reaches the acceptable circle C at the near/far limits D1, D2. The aperture A = f/N sets
# how fast the cone widens, so the smaller aperture (b) yields the larger depth of field.
n, R1, R2 = torch.tensor(1.5), torch.tensor(1.4), torch.tensor(1.4)   # tunable: glass, radii
U, N_a, N_b, C = 6.0, 3.2, 7.0, 0.10                                  # tunable: focus dist, two f-numbers, blur circle
f = (1.0 / ((n - 1) * (1 / R1 + 1 / R2))).item()
img = lambda D: f * D / (D - f)                  # thin-lens equation
s = img(U)                                       # sharp-image distance for the focus plane
def limits(N):                                   # aperture A=f/N and the near/far object-depth limits D1, D2
    A = f / N; r = C / A; vn, vf = s / (1 - r), s / (1 + r)
    D1 = f * vn / (vn - f); D2 = (f * vf / (vf - f)) if vf > f else 3.25 * U
    return A, D1, D2
(A_a, D1_a, D2_a), (A_b, D1_b, D2_b) = limits(N_a), limits(N_b)
print(f"f={f:.2f}, s={s:.2f}; DoF a:[{D1_a:.1f},{D2_a:.1f}]  b:[{D1_b:.1f},{D2_b:.1f}] (smaller aperture -> larger DoF)")

In [ ]:
# Figure 6.11 — two rows (apertures a, b). Each draws the near cone (solid) and far cone (faint)
# through the lens rim ±A to the sensor, with the aperture A, circle of confusion C, and D1/D2
# depth-of-field limits. Object depths are on a log-compressed axis so the far limits stay on-page.
import math
dark, gray, blue, red, cyan = COLORS["dark"], COLORS["gray"], COLORS["blue"], COLORS["red"], COLORS["cyan"]
x_obj, x_lens, x_sensor = -1.25, 2.15, 3.95
imsc = (x_sensor - x_lens) / s
xo = lambda D: x_obj - 1.65 * math.log(max(D, 1.05 * U) / U)     # log-compressed object axis
xi = lambda D: x_lens + imsc * img(D)                           # image position, mapped to the panel
yat = lambda p0, p1, x: p0[1] + (x - p0[0]) / (p1[0] - p0[0]) * (p1[1] - p0[1])
lx, rx, ly = lens_curves(0.0, 0.06, 0.62)

fig, ax = plt.subplots(figsize=(10.8, 8.0)); ax.set(xlim=(xo(D2_b) - 0.25, x_sensor + 1.15), ylim=(-3.35, 3.45)); ax.axis("off")
for x, lab, col in [(x_obj, "object plane", red), (x_sensor, "sensor plane", cyan)]:
    ax.plot([x, x], [-3.2, 3.2], color=col, lw=1.0, alpha=0.35, zorder=0)
    ax.text(x, 3.12, lab, fontsize=14, color=dark, ha="center", va="bottom")

for y0, A, D1, D2, tag in [(1.85, A_a, D1_a, D2_a, "a"), (-1.85, A_b, D1_b, D2_b, "b")]:
    h = 0.85 * A; lt, lb = (x_lens, y0 + h), (x_lens, y0 - h); xend = x_sensor + 0.28
    for xn, xf, lw, al, zo in [(xo(D1), xi(D1), 1.4, 1.0, 2), (xo(D2), xi(D2), 0.9, 0.38, 1)]:
        p, fi = (xn, y0), (xf, y0)
        for A0, B0 in [(p, lt), (p, lb), (lt, (xend, yat(lt, fi, xend))), (lb, (xend, yat(lb, fi, xend)))]:
            ax.plot([A0[0], B0[0]], [A0[1], B0[1]], color=blue, lw=lw, alpha=al, zorder=zo)
    draw_biconvex_lens(ax, lx + x_lens, rx + x_lens, ly + y0, face="white", lw=1.35, alpha=1.0)
    xA = x_lens - 0.20                                          # aperture bracket A = f/N
    ax.plot([xA, xA], [y0 - h, y0 + h], color=dark, lw=1.0, zorder=5)
    for yt in (y0 - h, y0 + h): ax.plot([xA - 0.08, xA + 0.08], [yt, yt], color=dark, lw=1.0, zorder=5)
    ax.text(xA - 0.50, y0, rf"$A^{tag}=\frac{{f}}{{N^{tag}}}$", fontsize=17, color=dark, ha="center", va="center")
    xC = x_sensor + 0.18                                        # circle of confusion C
    for yy in (y0 - 0.17, y0 + 0.17): ax.plot([x_sensor, xC], [yy, yy], color=gray, lw=1.0, ls=":", zorder=3)
    ax.plot([xC, xC], [y0 - 0.17, y0 + 0.17], color=gray, lw=1.0, zorder=3)
    ax.text(xC + 0.08, y0, r"$C$", fontsize=15, color=dark, ha="left", va="center")
    ax.text(xC + 0.42, y0, "circle of\nconfusion", fontsize=11, color=dark, ha="left", va="center")
    yb = y0 - 0.72; x1, x2 = xo(D1), xo(D2)                     # depth-of-field bracket + D1/D2 labels
    for xx in (x2, x1): ax.plot([xx, xx], [y0, yb], color=gray, lw=1.0, ls=":", zorder=0)
    ax.plot([x2, x1], [yb, yb], color=gray, lw=1.0, ls=":", zorder=0)
    ax.text((x1 + x2) / 2, yb + 0.07, "depth of field", fontsize=14, color=dark, ha="center", va="bottom")
    ax.text(x2 + 0.02, y0 - 0.10, rf"$D_2^{tag}$", fontsize=18, color=dark, ha="left", va="center")
    ax.text(x1 - 0.02, y0 - 0.10, rf"$D_1^{tag}$", fontsize=18, color=dark, ha="right", va="center")

ax.plot([x_obj, x_lens], [0.55, 0.55], color=gray, lw=1.0, ls=":", zorder=0)       # U bracket
ax.text((x_obj + x_lens) / 2, 0.63, r"$U$", fontsize=20, color=dark, ha="center", va="bottom")
ax.text(xo(D2_a) + 0.52, 2.07, r"$\approx \frac{CU}{f}$", fontsize=15, color=gray, ha="center", va="center")
ax.add_patch(FancyArrowPatch((xo(D2_a), -0.35), (xo(D2_b), -0.35), arrowstyle="-|>", mutation_scale=14, lw=1.4, color=red, zorder=6))
ax.text((xo(D2_a) + xo(D2_b)) / 2, -0.25, "larger depth of field", fontsize=12, color=red, ha="center", va="bottom")
xN = x_lens + 0.03
for s0, s1 in [(0.85 * A_a, 0.85 * A_b), (-0.85 * A_a, -0.85 * A_b)]:
    ax.add_patch(FancyArrowPatch((xN, -1.85 + s0), (xN, -1.85 + s1), arrowstyle="-|>", mutation_scale=12, lw=1.2, color=red, zorder=6))
ax.text(xN + 0.12, -1.85 - 0.18, "narrowed\naperture", fontsize=12, color=red, ha="left", va="center")
plt.savefig(IMAGES_DIR / "fig06_11.png", dpi=150, bbox_inches="tight")
plt.show()

The two similar-triangle pairs in Figure 6.11 yield expressions for $D_1$ and $D_2$ that, when solved and summed, give the exact depth of field:

$$D = \frac{2 N C U^2 f^2}{f^4 - N^2 C^2 U^2}.$$

For the practical regime $C \ll f/N$, the second term in the denominator drops out and the formula collapses to a clean proportionality:

$$D \approx \frac{2 N C U^2}{f^2}.$$

Depth of field is *linear* in $N$. Doubling $N$ doubles the in-focus range — but the light hitting the sensor falls as $1/N^2$, so the same scene needs four times the exposure. The next figure shows the trade-off on a real (synthetic) ruler.

**Figure 6.12 — Photographic depth of field as a function of aperture.** One sharp photograph of a ruler, blurred at each pixel by an amount proportional to its depth offset from the focal plane and scaled by $1/N$ — recreating the book's Figure 6.12 photographic demonstration. **(a) $f/2$:** narrow sharp band, blurred ends. **(b) $f/4$:** doubled DOF. **(c) $f/8$:** nearly the whole ruler sharp. (Book §6.3.1, Figure 6.12. Recreated by applying a Gaussian blur with $\sigma \propto 1/N$; the $f/2$ reference blur is calibrated visually.)

In [ ]:
# Figure 6.12 — photographic depth of field vs aperture. Starting from one sharp ruler photo,
# add depth-dependent defocus: the blur radius grows with aperture diameter (∝ 1/N), so a larger
# f-number N shrinks the blur disc and widens the sharp zone. Rows far from the focus row are
# "deeper" and blur more — the synthetic stand-in for the book's three real photographs.
img = torch.as_tensor(plt.imread(ASSETS_DIR / "ruler_single.png")[..., :3], dtype=torch.float32)
if img.max() > 1.5: img = img / 255.0
H, W, _ = img.shape
chw = img.permute(2, 0, 1)[None].to(DEVICE)
focus = 0.54 * (H - 1)                                          # tunable: which row is in focus
dist = (torch.arange(H).float() - focus).abs() / max(focus, H - 1 - focus)
strength = ((dist - 0.06) / 0.94).clamp(0, 1)[:, None].expand(H, W)     # per-row blur strength in [0,1]
print(f"ruler {H}x{W}; focus row {focus:.0f}; f/2,f/4,f/8 -> max sigma 8,4,2 px")

In [ ]:
# Figure 6.12 — blur each panel by its aperture, then show the three side by side. For each
# f-number, build a bank of Gaussian-blurred copies and pick, per pixel, the level matching that
# row's blur strength (max sigma ∝ 1/N, so f/2 blurs most and f/8 least).
LV = 9
ks = lambda s: 2 * int(-(-3.0 * s // 1)) + 1                    # odd Gaussian kernel size ~ 6·sigma
ri, ci = torch.arange(H)[:, None], torch.arange(W)[None, :]
def defocus(N):
    bank = [chw if s < 1e-6 else kornia.filters.gaussian_blur2d(chw, (ks(s), ks(s)), (s, s))
            for s in torch.linspace(0, 8.0 * 2.0 / N, LV).tolist()]
    bank = torch.stack([b[0].permute(1, 2, 0) for b in bank])   # (LV, H, W, 3)
    idx = (strength * (LV - 1)).round().long()                  # per-pixel blur level
    return bank[idx, ri, ci].clamp(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(9.8, 6.8))
for ax, N, lab in zip(axes, [2.0, 4.0, 8.0], ["(a) f/2.0", "(b) f/4.0", "(c) f/8.0"]):
    ax.imshow(defocus(N).cpu()); ax.axis("off")
    ax.text(0.5, -0.035, lab, transform=ax.transAxes, fontsize=13, color=COLORS["dark"], ha="center", va="top")
fig.subplots_adjust(left=0.02, right=0.98, top=0.99, bottom=0.08, wspace=0.08)
plt.savefig(IMAGES_DIR / "fig06_12.png", dpi=150, bbox_inches="tight")
plt.show()

The photographic trade-off — wider aperture, more light but shallower DOF — is what the formula $D \approx 2NCU^2/f^2$ makes quantitative. §6.3.2 turns the lens curvature inside out and asks what changes with a *concave* lens.

### 6.3.2 — Concave Lenses

The lens designed in §6.2 was *convex* — both surfaces bowing outward, focal length positive. Reverse the curvature on both surfaces and you get a *concave* lens, with both surfaces bowing inward. In the lensmaker's formula

$$\frac{1}{f} = (n-1)\left(\frac{1}{R_1} + \frac{1}{R_2}\right),$$

a concave surface contributes the same magnitude with the opposite sign, so $f$ comes out *negative*. The same paraxial geometry that focuses rays to a real point past a convex lens now bends them *away* from the axis at a concave lens — but the back-projection of those diverging rays still meets at a single point, on the *source side* of the lens. That point is the lens's *virtual* focal point.

**Figure 6.13 — Convex and concave thin-lens behavior.** **(a)** A convex lens with focal length $+f$: parallel rays from the left converge to a *real* focal point at distance $f$ past the lens. **(b)** A concave lens with focal length $-f$: parallel rays diverge after the lens, but their back-projections (dotted) meet at a *virtual* focal point at distance $f$ on the source side. **(c)** Same concave lens with a tilted parallel bundle: the virtual focal point shifts off-axis (cyan), just as the focal point in a convex lens shifts to image off-axis sources — the lensmaker's formula handles both cases with one sign change. (Book §6.3.2, Figure 6.13.)

In [ ]:
# Figure 6.13 — converging vs diverging lenses. A convex lens bends parallel rays to a real
# focus (a). A concave lens spreads them, and their backward extensions meet at a virtual focus
# on the entry side (b); tilting the incoming bundle only shifts that virtual focus off-axis (c).
f_conv, f_div, c, tilt = 0.82, 0.68, 0.34, -0.18     # tunable: convex/concave focal lengths, ray height, tilt slope
xL, xR = -1.45, 1.25
lx, rx, ly = lens_curves(torch.tensor(0.0), 0.11, 0.60)         # convex lens surfaces (shared helper)
yc = torch.linspace(-0.60, 0.60, 240)                          # concave lens body: thin waist, thick edges
xcc = 0.035 + 0.055 * (1 - torch.sqrt((1 - (yc / 0.60) ** 2).clamp(min=0)))
print(f"convex focus +{f_conv}, concave virtual focus {-f_div}, tilt slope {tilt}")

In [ ]:
# Figure 6.13 — three panels. Solid rays are real; dashed rays are the backward extensions that
# locate a virtual focus. Convex uses the shared biconvex lens; concave is a thin-waist body.
dark, blue, cyan, pg = COLORS["dark"], COLORS["blue"], COLORS["cyan"], COLORS["panel"]
convex = list(zip(rx.tolist(), ly.tolist())) + list(zip(lx.flip(0).tolist(), ly.flip(0).tolist()))
concave = list(zip(xcc.tolist(), yc.tolist())) + list(zip((-xcc).flip(0).tolist(), yc.flip(0).tolist()))
lens = lambda ax, poly: ax.add_patch(Polygon(poly, closed=True, facecolor="white", edgecolor=dark, lw=1.2, zorder=3))
seg = lambda ax, p0, p1, col, ls="-", lw=1.15, z=2: ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=col, ls=ls, lw=lw, zorder=z)
dash = (0, (2.2, 2.2))

fig, axes = plt.subplots(1, 3, figsize=(9.2, 3.0))
for ax in axes: ax.set(xlim=(xL, xR), ylim=(-0.82, 0.82)); ax.axis("off")
# (a) convex: parallel rays in -> converge to a real focus
lens(axes[0], convex); axes[0].scatter([f_conv], [0], s=10, color=blue, zorder=5)
for y in (c, 0, -c): seg(axes[0], (xL, y), (0, y), blue)
for y in (c, -c): seg(axes[0], (0, y), (f_conv, 0), blue)
seg(axes[0], (0, 0), (xR, 0), blue)
# (b) concave: parallel rays diverge; dashed back-projection meets at the virtual focus
lens(axes[1], concave); axes[1].scatter([-f_div], [0], s=10, color=blue, zorder=5)
for y in (c, 0, -c):
    seg(axes[1], (xL, y), (0, y), blue)
    if abs(y) > 1e-6:
        seg(axes[1], (-f_div, 0), (0, y), blue, ls=dash, lw=1.0, z=1)
        seg(axes[1], (0, y), (xR, y + y / f_div * xR), blue)
    else: seg(axes[1], (0, 0), (xR, 0), blue)
# (c) concave with a tilted bundle -> off-axis virtual focus (distinct colour)
lens(axes[2], concave); axes[2].scatter([-f_div], [-tilt * f_div], s=10, color=cyan, zorder=5)
seg(axes[2], (xL, 0), (xR, 0), blue, z=1)
for y in (c, 0, -c): seg(axes[2], (xL, y + tilt * xL), (0, y), cyan)
for y in (c, -c): seg(axes[2], (-f_div, -tilt * f_div), (0, y), cyan, ls=dash, lw=1.0, z=1)
for y in (c, 0, -c): seg(axes[2], (0, y), (xR, y + (tilt + y / f_div) * xR), cyan)
for ax, t in zip(axes, "abc"): ax.text(0, -0.73, f"({t})", fontsize=13, color=pg, style="italic", ha="center")
fig.subplots_adjust(left=0.03, right=0.99, top=0.98, bottom=0.08, wspace=0.13)
plt.savefig(IMAGES_DIR / "fig06_13.png", dpi=150, bbox_inches="tight")
plt.show()

A concave lens alone can't form a real image — it diverges every bundle it sees. But paired with a convex lens, that diverging behavior becomes useful: if the concave lens sits at the convex lens's focal point, the two together turn a converging bundle back into parallel rays — at a *different* angle than they entered. That angular amplification is the principle behind the Galilean telescope, which §6.3.3 builds next.

### 6.3.3 — Lenses in a Telescope

A convex lens and a concave lens together form a *Galilean telescope*, named after the one Galileo built in 1609. The construction is simple: place a convex lens (lens 1, focal length $f_1$, long) with a concave lens (lens 2, focal length $f_2$, shorter) such that they share a common focal point — lens 1 is distance $f_1$ to the left of that shared point, lens 2 is distance $f_2$ to the same side. The lenses sit $f_1 - f_2$ apart. In that configuration, parallel rays entering lens 1 converge toward the shared focal point, and lens 2 — intercepting them before they reach it — refracts them back into a parallel bundle. The output bundle is parallel like the input, but compressed into a smaller cross-section.

What makes it a *telescope* is what happens when the input direction tilts.

**Figure 6.14 — Galilean telescope geometry.**

**(a) Parallel input, parallel output.** Three parallel rays enter lens 1 from the left, converge toward the shared focal point, and are intercepted by lens 2 before reaching it — refracting back into a parallel bundle on the right. The output rays are parallel like the input, but closer together, compressed into the smaller exit aperture.

**(b) Tilted input, angular magnification.** When the input bundle tilts at angle $\delta_i$ from the optical axis, the chief ray through lens 1's center reaches the shared focal point at height $d = f_1 \delta_i$ above the axis (small-angle approximation, property 3 of §6.3). The same point $d$ is at distance $f_2$ from lens 2, which refracts the bundle into a parallel output at angle $\delta_o = d / f_2$ from the axis. The two similar triangles share the height $d$ but have different base lengths $f_1$ and $f_2$, giving the magnification $M = \delta_o / \delta_i = f_1 / f_2$. (Book §6.3.3, Figure 6.14.)

In [ ]:
# Figure 6.14 — a Galilean telescope: convex objective (lens 1, focal length f1) and concave
# eyepiece (lens 2, f2) sharing a focal point at f1, with lens 2 placed f2 before it. (a) A
# parallel bundle in leaves parallel but narrower. (b) A bundle tilted by δ_i leaves at the
# larger angle δ_o = d/f2 with d = f1·δ_i, giving angular magnification M = δ_o/δ_i = f1/f2.
f1, f2 = 3.9, 1.35                                 # tunable: objective / eyepiece focal lengths
ray_h, di = 0.62, torch.deg2rad(torch.tensor(5.5))   # tunable: input ray height (a); input tilt (b)
xL, x1, x2, xF = -2.9, 0.0, f1 - f2, f1            # left edge, lens 1, lens 2, shared focal point
xR = xF + 1.35
print(f"f1={f1}, f2={f2} -> magnification M=f1/f2={f1/f2:.2f}; δ_i={torch.rad2deg(di):.1f}deg")

In [ ]:
# Figure 6.14 — two stacked panels. Lens 1 is a tall biconvex, lens 2 a biconcave; solid rays
# are real and dashed rays back-project to the shared focal point. Both share the f1/f2 brackets.
dark, blue, pg, violet = COLORS["dark"], COLORS["blue"], COLORS["panel"], "#c06df2"
u = torch.linspace(-1, 1, 240); s = torch.sqrt((1 - u ** 2).clamp(min=0))
lens1 = list(zip((x1 + 0.14 * s).tolist(), (1.65 * u).tolist())) + list(zip((x1 - 0.14 * s).flip(0).tolist(), (1.65 * u).flip(0).tolist()))
w = 0.18 * (0.30 + 0.70 * (1 - s))
lens2 = list(zip((x2 + w).tolist(), (0.92 * u).tolist())) + list(zip((x2 - w).flip(0).tolist(), (0.92 * u).flip(0).tolist()))
lp = lambda ax, poly: ax.add_patch(Polygon(poly, closed=True, facecolor="white", edgecolor=dark, lw=1.4, zorder=4))
seg = lambda ax, a, b, col, ls="-", lw=1.1, z=3, al=1.0: ax.plot([a[0], b[0]], [a[1], b[1]], color=col, ls=ls, lw=lw, zorder=z, alpha=al)
dash = (0, (2, 2)); t = torch.linspace(0, 1, 60)
arc = lambda cx, cy, r, a0, a1: [(cx + r * torch.cos(a0 + (a1 - a0) * t)).tolist(), (cy + r * torch.sin(a0 + (a1 - a0) * t)).tolist()]
def brackets(ax, yb):
    for xa, xb, y, lab in [(x1, xF, yb, r"$f_1$"), (x2, xF, yb + 0.30, r"$f_2$")]:
        seg(ax, (xa, y), (xb, y), dark, lw=1.0)
        for xe in (xa, xb): seg(ax, (xe, y - 0.07), (xe, y + 0.07), dark, lw=1.0)
        ax.text((xa + xb) / 2, y + 0.12, lab, fontsize=16, color=dark, ha="center", va="bottom")
def frame(ax, tag):
    ax.text(x1 + 0.20, 1.75, "lens 1", fontsize=15, color=blue, ha="center")
    ax.text(x2 + 0.20, 1.20, "lens 2", fontsize=15, color=blue, ha="center")
    ax.text((xL + xR) / 2, -2.10, f"({tag})", fontsize=17, color=pg, style="italic", ha="center")
    ax.set(xlim=(xL - 0.25, xR + 0.25), ylim=(-2.25, 2.05)); ax.axis("off")

fig, axes = plt.subplots(2, 1, figsize=(11.5, 8.5))
# (a) parallel in -> narrower parallel out
ax = axes[0]; seg(ax, (xL, 0), (xR, 0), blue, lw=0.9, z=0, al=0.55); lp(ax, lens1); lp(ax, lens2)
for y in (ray_h, 0, -ray_h):
    y2 = y * (f2 / f1)
    seg(ax, (xL, y), (x1, y), blue); seg(ax, (x1, y), (x2, y2), blue); seg(ax, (x2, y2), (xR, y2), blue)
    seg(ax, (x2, y2), (xF, 0), blue, ls=dash, lw=1.0, z=2)
ax.scatter([xF], [0], s=38, color=blue, zorder=5); brackets(ax, -1.55); frame(ax, "a")
# (b) tilted in -> angular magnification
ax = axes[1]
mi = -torch.tan(di).item(); yF = 0.03 + mi * (xF - x1); d = -yF
do = torch.atan(torch.tensor(d / f2)).item(); mo = -torch.tan(torch.tensor(do)).item()
bundle = lambda y1: ((xL, y1 + mi * (xL - x1)), (x1, y1), (x2, y1 + (yF - y1) / (xF - x1) * x2), (xR, y1 + (yF - y1) / (xF - x1) * x2 + mo * (xR - x2)))
seg(ax, (xL, 0), (xR, 0), blue, lw=1.0, z=1, al=0.45); lp(ax, lens1); lp(ax, lens2)
for y1, lw, al in [(0.95, 2.0, 1.0), (0.03, 1.0, 0.7)]:
    pL, p1, p2, pR = bundle(y1)
    for a, b in [(pL, p1), (p1, p2), (p2, pR)]: seg(ax, a, b, violet, lw=lw, al=al)
    seg(ax, p2, (xF, yF), violet, ls=dash, lw=1.0, z=2, al=0.85)
ax.scatter([xF], [0], s=44, color=blue, zorder=5)
ax.annotate("", xy=(xF, yF), xytext=(xF, 0), arrowprops=dict(arrowstyle="->", lw=1.0, color=dark))
ax.text(xF + 0.07, yF / 2, r"$d$", fontsize=16, color=dark, ha="left", va="center")
for y1 in (0.95, 0.03):                                     # δ_i arc on each input ray
    yr = y1 + mi * (xL - x1) - 0.10
    seg(ax, (xL + 0.04, yr), (xL + 0.42, yr), dark, lw=1.0)
    ax.plot(*arc(xL + 0.32, yr, 0.34, -di.item(), 0.0), color=dark, lw=1.0)
    ax.text(xL + 0.12, y1 + mi * (xL - x1) - 0.30, r"$\delta_i$", fontsize=18, color=dark, ha="right", va="center")
seg(ax, (xF + 0.15, -0.02), (xF + 0.57, -0.02), dark, lw=1.0)   # δ_o arc at the focal point
ax.plot(*arc(xF + 0.15, -0.02, 0.42, -do, 0.0), color=dark, lw=1.0)
ax.text(xF + 0.62, -0.20, r"$\delta_o$", fontsize=18, color=dark, ha="left", va="center")
brackets(ax, -1.55); frame(ax, "b")
fig.subplots_adjust(left=0.04, right=0.98, top=0.98, bottom=0.04, hspace=0.18)
plt.savefig(IMAGES_DIR / "fig06_14ab.png", dpi=150, bbox_inches="tight")
plt.show()

With $f_1 > f_2$ the telescope magnifies. Galileo built his at $M \approx 30$ by pairing a long convex objective with a short concave eyepiece; the book's Figs 6.15 and 6.16 show a cardboard recreation ($M \approx 27$ from $f_1 = 500$ mm, $f_2 = 18$ mm) and the moon through it, alongside Galileo's own lunar drawings. Those photographs aren't reproduced here.

With this, §6.3 is complete. The chapter has worked from Snell's law through the lensmaker's formula and used it for imaging, depth of field, concave lenses, and the telescope.